In [1]:
import os
import sys 
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql import SparkSession







In [2]:
PROJECT_DIR = Path(
    r"D:\projects\AI_Monitoring_Project"
)

ETL_DIR = PROJECT_DIR / "advanced" / "ETL"

WAREHOUSE_DIR = ETL_DIR / "warehouse"
METASTORE_DIR = ETL_DIR / "metastore_db"
SPARK_TEMP_DIR = ETL_DIR / ".spark-temp"

WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)
SPARK_TEMP_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
JAVA_HOME = Path(
    r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
)

HADOOP_HOME = Path(
    r"C:\hadoop\hadoop-3.5.0"
)

os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["HADOOP_HOME"] = str(HADOOP_HOME)

os.environ["PATH"] = (
    str(JAVA_HOME / "bin")
    + os.pathsep
    + str(HADOOP_HOME / "bin")
    + os.pathsep
    + os.environ.get("PATH", "")
)

os.environ["PYSPARK_PYTHON"] = sys.executable

In [4]:
METASTORE_URL = (
    f"jdbc:derby:;databaseName="
    f"{METASTORE_DIR.as_posix()};create=true"
)

In [5]:
spark = (
    SparkSession.builder
    .appName("AI_Monitoring_Queries")
    .master("local[*]")

    .config(
        "spark.sql.warehouse.dir",
        WAREHOUSE_DIR.as_uri()
    )

    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        METASTORE_URL
    )

    .config(
        "spark.local.dir",
        str(SPARK_TEMP_DIR)
    )

    .enableHiveSupport()
    .getOrCreate()
)

print("Spark initialized successfully!")
print("Spark version:", spark.version)
print("Python executable:", sys.executable)
print("Warehouse:", spark.conf.get("spark.sql.warehouse.dir"))

Spark initialized successfully!
Spark version: 4.2.0
Python executable: d:\projects\AI_Monitoring_Project\.venv\Scripts\python.exe
Warehouse: file:/D:/projects/AI_Monitoring_Project/advanced/ETL/warehouse


In [6]:
spark.sql("SHOW TABLES IN gold").show(truncate=False)

+---------+----------------------------+-----------+
|namespace|tableName                   |isTemporary|
+---------+----------------------------+-----------+
|gold     |employee_directory          |false      |
|gold     |monitoring_detection_details|false      |
|gold     |monitoring_events           |false      |
|gold     |restricted_access           |false      |
+---------+----------------------------+-----------+



In [7]:
monitoring_df = spark.table("gold.monitoring_detection_details")
monitoring_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- Zone: string (nullable = true)
 |-- camera_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- detector_type: string (nullable = true)
 |-- detection_source: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- plate_number: string (nullable = true)
 |-- authorized: boolean (nullable = true)



In [8]:
print("total monitoring records: ", monitoring_df.count())
display(monitoring_df.limit(10).toPandas())

total monitoring records:  36473


,timestamp,Zone,camera_id,event_id,detector_type,detection_source,employee_id,plate_number,authorized
0,2026-10-28 12:09:58.634651,A,001,camera.person.frames.dev:0:2637,synthetic,person,EMP0029,None,True
1,2026-10-28 12:09:58.634651,A,001,camera.person.frames.dev:0:2637,synthetic,person,EMP0072,None,True
2,2026-10-28 12:09:58.634651,A,002,camera.person.frames.dev:0:2638,synthetic,person,EMP0088,None,True
3,2026-10-28 12:09:59.691376,A,001,camera.person.frames.dev:0:2640,synthetic,person,EMP0029,None,True
4,2026-10-28 12:09:59.691376,A,001,camera.person.frames.dev:0:2640,synthetic,person,EMP0072,None,True
5,2026-10-28 12:09:59.691376,A,002,camera.person.frames.dev:0:2641,synthetic,person,EMP0088,None,True
6,2026-10-28 12:10:00.702102,A,001,camera.person.frames.dev:0:2643,synthetic,person,EMP0011,None,True
7,2026-10-28 12:10:00.702102,A,001,camera.person.frames.dev:0:2643,synthetic,person,EMP0029,None,True
8,2026-10-28 12:10:00.702102,A,002,camera.person.frames.dev:0:2644,synthetic,person,EMP0057,None,True
9,2026-10-28 12:10:00.703101,B,003,camera.person.frames.dev:0:2645,synthetic,person,EMP0015,None,True


In [9]:
employee_activity_df = (
    monitoring_df
    .filter(F.col("employee_id").isNotNull())
    .groupBy("employee_id")
    .count()
    .orderBy("count", ascending = False)
)

display(employee_activity_df.toPandas())
employee_row = employee_activity_df.first()
selected_employee_id = employee_row["employee_id"]


,employee_id,count
0,EMP0077,1334
1,EMP0039,1332
2,EMP0003,1323
3,EMP0021,1297
4,EMP0036,1266
5,EMP0020,1238
6,EMP0095,1207
7,EMP0059,1207
8,EMP0046,1180
9,EMP0029,1149


In [10]:
employee_row = employee_activity_df.first()
selected_employee_id = employee_row["employee_id"]
print("selected employee is: ", selected_employee_id)

selected employee is:  EMP0077


In [11]:
employee_history_df = (
    monitoring_df

    .filter(
        F.col("employee_id") == selected_employee_id
    )

    .select(
        "timestamp",
        "employee_id",
        "Zone",
        "camera_id",
        "event_id",
        "detector_type",
        "detection_source",
        "plate_number",
        "authorized"
    )

    .orderBy(
        F.col("timestamp").asc(),
        F.col("event_id").asc(),
        F.col("camera_id").asc()
    )
)

display(
    employee_history_df.limit(50).toPandas()
)

,timestamp,employee_id,Zone,camera_id,event_id,detector_type,detection_source,plate_number,authorized
0,2026-10-28 12:10:00.703101,EMP0077,B,003,camera.person.frames.dev:0:2645,synthetic,person,None,True
1,2026-10-28 12:10:01.716779,EMP0077,B,003,camera.person.frames.dev:0:2648,synthetic,person,None,True
2,2026-10-28 12:10:02.728678,EMP0077,B,003,camera.person.frames.dev:0:2651,synthetic,person,None,True
3,2026-10-28 12:10:03.739529,EMP0077,B,003,camera.person.frames.dev:0:2654,synthetic,person,None,True
4,2026-10-28 12:10:04.751150,EMP0077,B,003,camera.person.frames.dev:0:2657,synthetic,person,None,True
5,2026-10-28 12:10:05.762638,EMP0077,B,003,camera.person.frames.dev:0:2660,synthetic,person,None,True
6,2026-10-28 12:10:06.773636,EMP0077,B,003,camera.person.frames.dev:0:2663,synthetic,person,None,True
7,2026-10-28 12:10:07.786793,EMP0077,B,003,camera.person.frames.dev:0:2666,synthetic,person,None,True
8,2026-10-28 12:10:08.798281,EMP0077,B,003,camera.person.frames.dev:0:2669,synthetic,person,None,True
9,2026-10-28 12:10:09.808504,EMP0077,B,003,camera.person.frames.dev:0:2672,synthetic,person,None,True


In [12]:
import sys
from pathlib import Path

ETL_DIR = Path(
    r"D:\projects\AI_Monitoring_Project\advanced\ETL"
)

if str(ETL_DIR) not in sys.path:
    sys.path.insert(0, str(ETL_DIR))

In [14]:
from analytics.services.person_monitoring import (
    PersonMonitoringService
)

person_monitoring = PersonMonitoringService(
    spark=spark
)

In [15]:
employee_row = (
    spark.table("gold.monitoring_detection_details")
    .filter(F.col("employee_id").isNotNull())
    .select("employee_id")
    .first()
)

if employee_row is None:
    raise ValueError("No employee observations found.")

selected_employee_id = employee_row["employee_id"]

employee_history_df = (
    person_monitoring.get_employee_history(
        employee_id=selected_employee_id,
        limit=50
    )
)

display(employee_history_df.toPandas())

,timestamp,employee_id,Zone,camera_id,event_id,detector_type,detection_source,plate_number,authorized
0,2026-10-28 12:09:58.634651,EMP0029,A,001,camera.person.frames.dev:0:2637,synthetic,person,None,True
1,2026-10-28 12:09:59.691376,EMP0029,A,001,camera.person.frames.dev:0:2640,synthetic,person,None,True
2,2026-10-28 12:10:00.702102,EMP0029,A,001,camera.person.frames.dev:0:2643,synthetic,person,None,True
3,2026-10-28 12:10:01.712931,EMP0029,A,001,camera.person.frames.dev:0:2646,synthetic,person,None,True
4,2026-10-28 12:10:02.726185,EMP0029,A,001,camera.person.frames.dev:0:2649,synthetic,person,None,True
5,2026-10-28 12:10:03.738030,EMP0029,A,001,camera.person.frames.dev:0:2652,synthetic,person,None,True
6,2026-10-28 12:10:04.748573,EMP0029,A,001,camera.person.frames.dev:0:2655,synthetic,person,None,True
7,2026-10-28 12:10:05.760503,EMP0029,A,001,camera.person.frames.dev:0:2658,synthetic,person,None,True
8,2026-10-28 12:10:06.773109,EMP0029,A,001,camera.person.frames.dev:0:2661,synthetic,person,None,True
9,2026-10-28 12:10:07.785149,EMP0029,A,001,camera.person.frames.dev:0:2664,synthetic,person,None,True


In [16]:
assert (
    employee_history_df
    .filter(
        F.col("employee_id") != selected_employee_id
    )
    .count() == 0
)

print("Employee history service validated!")
print("Employee:", selected_employee_id)
print("Returned records:", employee_history_df.count())

Employee history service validated!
Employee: EMP0029
Returned records: 50
